<a href="https://colab.research.google.com/github/NatNataNaty/INTELIGENCIA_ARTIFICIAL/blob/main/Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
from matplotlib import pyplot
from scipy import optimize
from google.colab import drive
from sklearn.decomposition import PCA # Necesario para graficar 30,000 dimensiones
import ipywidgets as widgets
from IPython.display import display

# 1. MONTAR DRIVE
drive.mount('/content/drive')

# 2. CARGA DE DATOS (Simulación de n>40, m>30000)
# Para tu caso real usa: data = np.loadtxt('/ruta/al/dataset.txt', delimiter=',')
path = '/content/drive/MyDrive/Inteligencia Artificial/Laboratorio 2/Dallas Police Arrests.csv'

n_ejemplos = 100
m_features = 30000
X_raw = np.random.randn(n_ejemplos, m_features)
y = np.random.randint(0, 2, n_ejemplos)

print(f"Dataset cargado: {X_raw.shape[0]} ejemplos y {X_raw.shape[1]} características.")

# 3. FUNCIONES DEL CUADERNILLO
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def featureNormalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    # Evitar división por cero
    sigma[sigma == 0] = 1
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

def costFunction(theta, X, y):
    m = y.size
    h = sigmoid(X.dot(theta))
    epsilon = 1e-15
    J = (1 / m) * np.sum(-y.dot(np.log(h + epsilon)) - (1 - y).dot(np.log(1 - h + epsilon)))
    grad = (1 / m) * (h - y).dot(X)
    return J, grad

# 4. PREPARACIÓN Y ENTRENAMIENTO
X_norm, mu, sigma = featureNormalize(X_raw)
m, n = X_norm.shape
X = np.concatenate([np.ones((m, 1)), X_norm], axis=1)

initial_theta = np.zeros(n + 1)
options = {'maxiter': 100}

print("Entrenando... (esto puede tomar un minuto debido a las 30,000 columnas)")
res = optimize.minimize(costFunction, initial_theta, (X, y), jac=True, method='TNC', options=options)
theta_optimizado = res.x

# 5. GRÁFICA: REDUCCIÓN DE DIMENSIONALIDAD (PCA)
# Como no podemos graficar 30,000 dimensiones, usamos PCA para proyectarlas a 2D
def plotHighDim(X_data, y_label):
    pca = PCA(n_components=2)
    X_reduced = pca.fit_transform(X_data[:, 1:]) # Quitamos el intercepto para PCA

    pos = y_label == 1
    neg = y_label == 0

    pyplot.figure(figsize=(10, 7))
    pyplot.plot(X_reduced[pos, 0], X_reduced[pos, 1], 'k*', lw=2, ms=10, label='Clase 1')
    pyplot.plot(X_reduced[neg, 0], X_reduced[neg, 1], 'ko', mfc='y', ms=8, label='Clase 0')
    pyplot.title('Visualización del Dataset de 30,000 columnas usando PCA')
    pyplot.xlabel('Componente Principal 1')
    pyplot.ylabel('Componente Principal 2')
    pyplot.legend()
    pyplot.grid(True)
    pyplot.show()

plotHighDim(X, y)

# 6. INTERFAZ INTERACTIVA (VENTANILLA)
# --- VENTANILLA MODIFICADA PARA UN SOLO DATO ---

input_text = widgets.Text(
    value='',
    placeholder='Ingresa un solo valor numérico...',
    description='Valor X1:',
    style={'description_width': 'initial'}
)
btn_predecir = widgets.Button(description="Predecir y", button_style='success')
resultado_label = widgets.HTML(value="<b>Resultado aparecerá aquí</b>")

def on_predict_clicked(b):
    try:
        # 1. Leer el único dato ingresado
        dato_unico = float(input_text.value)

        # 2. Crear un vector de 30,000 campos lleno de ceros
        X_input = np.zeros(m_features)

        # 3. Asignar el dato ingresado a la primera posición
        X_input[0] = dato_unico

        # 4. Normalizar (usando mu y sigma guardados)
        X_norm_input = (X_input - mu) / sigma

        # 5. Agregar el 1 del intercepto
        X_final = np.append([1], X_norm_input)

        # 6. Calcular probabilidad y clase
        prob = sigmoid(np.dot(X_final, theta_optimizado))
        clase = 1 if prob >= 0.5 else 0

        color = "#d4edda" if clase == 1 else "#f8d7da"
        resultado_label.value = f"""
        <div style='background-color:{color}; padding:15px; border-radius:5px;'>
            <b>Predicción:</b> Clase {clase} <br>
            <b>Probabilidad:</b> {prob:.4f}
        </div>
        """
    except ValueError:
        resultado_label.value = "<b style='color:red'>Por favor, ingresa un número válido.</b>"

btn_predecir.on_click(on_predict_clicked)

display(input_text, btn_predecir, resultado_label)